<a href="https://colab.research.google.com/github/raunit1610/collab-notebook-repo/blob/ipl/IPL_Data_Fetch_(API's).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#API's (2022 - 2025)

##Import Necessary Library

In [ ]:
!pip install snowflake-connector-python
!pip3 install boto3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 3.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of pyopenssl to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 43.0.3
    Uninstalling cryptography-43.0.3:
      Successfully uninstalled cryptography-43.0.3
  Attempting uninstall: pyOpenSSL
    Found ex

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import re
import pandas as pd
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
from io import StringIO
from datetime import datetime
import boto3
import time

##Snowflake Connection

In [ ]:
conn = snowflake.connector.connect(
    user="IPLanalyst",
    password="abcd",
    account="EAPQGUP-JC73769",
    warehouse="EXTERNAL_WH",
    database="IPL_RAW_DATABASE",
    schema="PUBLIC",
    role="GCollab"
)
cur = conn.cursor()
print("Connected to Snowflake!")

Connected to Snowflake!


##Necessary Functions


In [ ]:
def convert_to_json(txt, prefix):
    # Remove the prefix and the final ")"
    if txt.startswith(prefix):
        txt = txt[len(prefix):-2]
    try:
        return json.loads(txt)
    except Exception as e:
        print(f"JSON parsing failed : {e}")
        return None

def fetch_webpages(url):
  try:
    response = requests.get(url)
    time.sleep(1.0)
    if response.status_code == 200:
        print("Successfully fetched the webpage")
        return response
    else:
        print("Failed to fetch the webpage")
        print("Status code:", response.status_code)
        return None
  except requests.exceptions.RequestException as e:
      print(f"An error occurred: {e}")
  return None

def fetch_data_y(url, parameter, outlier, objects):
    try:
        response = requests.get(url.format(year = parameter))
        time.sleep(1.0)
        if response.status_code == 200:
            txt = response.text.strip()
            data = convert_to_json(txt, outlier)
            lists = data.get(objects, [])

            if not isinstance(lists, list):
              print(f"Didn't Found Object data in URL")
              return []
            return lists
        else:
            print(f"Failed to fetch data from {url}")
            return []
    except Exception as e:
        print(f"An error occurred: {e}")
        return []

def fetch_data_m(url, parameter, outlier, objects):
    try:
        response = requests.get(url.format(MatchID = parameter))
        time.sleep(1.0)
        if response.status_code == 200:
            txt = response.text.strip()
            data = convert_to_json(txt, outlier)
            lists = data.get(objects, [])

            if not isinstance(lists, list):
              print(f"Didn't Found Object data in URL")
              return []
            return lists
        else:
            print(f"Failed to fetch data from {url}")
            return []
    except Exception as e:
        print(f"An error occurred: {e}")
        return []

def innings(match_id,base_innings_url,Innings):
   innings_url = base_innings_url.format(MatchID=match_id)
   innings_resp = requests.get(innings_url)
   time.sleep(1.0)
   if innings_resp.status_code == 200:
      txt = innings_resp.text.strip()
      data = convert_to_json(txt, "onScoring(")
      return data

   else:
      print(f"Could not fetch {Innings} for matchID {match_id}")

def yearslist(year_url):
   years_url = year_url
   years_resp = requests.get(years_url)
   time.sleep(1.0)
   if years_resp.status_code == 200:
      txt = years_resp.text.strip()
      data = convert_to_json(txt, "oncomptetion(")
      return data

   else:
      print(f"Could not fetch Years Details")

def squad_fun(match_id,base_url):
   squad_url = base_url.format(MatchID=match_id)
   squad_resp = requests.get(squad_url)
   time.sleep(1.0)
   if squad_resp.status_code == 200:
      txt = squad_resp.text.strip()
      data = convert_to_json(txt, "onsquad(")
      return data

   else:
      print(f"Could not fetch Squad for matchID {match_id}")

##Shema Defining Functions

In [ ]:
#Carriers
#Events Record Schema
def extract_seasonsDetail(recordA,recordB):
  SeasonsDetail = {
      "CompetitionID": recordB.get("CompetitionID"),
      "CompetitionName": recordB.get("CompetitionName"),
      "DivisionID": recordA.get("DivisionID"),
      "DivisionName": recordA.get("DivisionName"),
      "MappingCompName": recordB.get("MappingCompName"),
      "CompetitionType": recordB.get("CompetitionType"),
      "CodingType": recordB.get("CodingType"),
      "feedsource": recordB.get("feedsource"),
      "statsFeed": recordB.get("statsFeed"),
      "statsCoding": recordB.get("statsCoding"),
      "statsCID": recordB.get("statsCID"),
      "SeasonID": recordA.get("SeasonID"),
      "SeasonName": recordA.get("SeasonName"),
      "timezone": recordB.get("timezone"),
      "live": recordB.get("live"),
      "fixtures": recordB.get("fixtures"),
      "completed": recordB.get("completed"),
      "Standings": recordB.get("Standings"),
      "TeamType": recordB.get("TeamType")
  }
  return SeasonsDetail

#Players Batting Carrier Schema
def extract_battingCarrier(record):
    battingCarrier = {
        "PlayerId": record.get("PlayerId"),
        "PlayerName": record.get("PlayerName"),
        "Year": record.get("Year"),
        "TeamName": record.get("TeamName"),
        "Matches": record.get("Matches"),
        "Innings": record.get("Innings"),
        "Runs": record.get("Runs"),
        "Balls": record.get("Balls"),
        "HighestScore": record.get("HighestScore"),
        "Fifties": record.get("Fifties"),
        "Hundreds": record.get("Hundreds"),
        "Sixes": record.get("Sixes"),
        "Fours": record.get("Fours"),
        "StrikeRate": record.get("StrikeRate"),
        "BattingAvg": record.get("BattingAvg"),
        "CompetitionId": record.get("CompetitionId"),
        "CompetitionName": record.get("CompetitionName"),
        "TeamShortName": record.get("TeamShortName"),
        "ClientPlayerID": record.get("ClientPlayerID"),
        "NotOuts": record.get("NotOuts"),
        "Catches": record.get("Catches"),
        "Stumpings": record.get("Stumpings"),
        "StatsType": record.get("StatsType")
    }
    return battingCarrier

#Players Bowling Carrier Schema
def extract_bowlingCarrier(record):
    bowlingCarrier = {
        "PlayerId": record.get("PlayerId"),
        "PlayerName": record.get("PlayerName"),
        "Year": record.get("Year"),
        "TeamName": record.get("TeamName"),
        "Matches": record.get("Matches"),
        "Innings": record.get("Innings"),
        "Overs": record.get("Overs"),
        "Balls": record.get("Balls"),
        "Runs": record.get("Runs"),
        "Wickets": record.get("Wickets"),
        "BBM": record.get("BBM"),
        "Average": record.get("Average"),
        "Econ": record.get("Econ"),
        "StrikeRate": record.get("StrikeRate"),
        "FourWkts": record.get("FourWkts"),
        "FiveWkts": record.get("FiveWkts"),
        "StatsType": record.get("StatsType")
    }
    return bowlingCarrier

#Seasons
# Match Summary Schema
def extract_matchsummary(schedrecord,summrecord):
    matchSummary = {
        "CompetitionID":schedrecord.get("CompetitionID"),
        "CompetitionName":summrecord.get("CompetitionName"),
        "GroundID":schedrecord.get("GroundID"),
        "GroundName":schedrecord.get("GroundName"),
        "city":schedrecord.get("city"),
        "Temperature":schedrecord.get("Temperature"),
        "MatchID":schedrecord.get("MatchID"),
        "MatchName":schedrecord.get("MatchName"),
        "MatchType":summrecord.get("MatchType"),
        "MATCH_COMMENCE_START_DATE":schedrecord.get("MATCH_COMMENCE_START_DATE"),
        "TimeZone":schedrecord.get("TimeZone"),
        "timezone1":schedrecord.get("timezone1"),
        "MatchDate":schedrecord.get("MatchDate"),
        "GMTMatchDate":schedrecord.get("GMTMatchDate"),
        "MatchEndDate":schedrecord.get("MatchEndDate"),
        "GMTMatchEndDate":schedrecord.get("GMTMatchEndDate"),
        "MatchTime":schedrecord.get("MatchTime"),
        "GMTMatchTime":schedrecord.get("GMTMatchTime"),
        "MatchEndTime":schedrecord.get("MatchEndTime"),
        "GMTMatchEndTime":schedrecord.get("GMTMatchEndTime"),
        "MATCH_NO_OF_OVERS":schedrecord.get("MATCH_NO_OF_OVERS"),
        "RevisedOver":schedrecord.get("RevisedOver"),
        "RevisedTarget":schedrecord.get("RevisedTarget"),
        "CurrentInnings":schedrecord.get("CurrentInnings"),
        "IsMatchEnd":summrecord.get("IsMatchEnd"),
        "MOMPlayerId":schedrecord.get("MOMPlayerId"),
        "MOM":schedrecord.get("MOM"),
        "MOM_TYPE":schedrecord.get("MOM_TYPE"),
        "MOMRuns":schedrecord.get("MOMRuns"),
        "MOMBalls":schedrecord.get("MOMBalls"),
        "MOMWicket":schedrecord.get("MOMWicket"),
        "MOMRC":schedrecord.get("MOMRC"),
        "MOMImage":schedrecord.get("MOMImage"),
        "MatchOrder":schedrecord.get("MatchOrder"),
        "KO":schedrecord.get("KO"),
        "RemainingBalls":summrecord.get("RemainingBalls"),
        "GroundUmpire1ID":schedrecord.get("GroundUmpire1ID"),
        "Umpire1Name":summrecord.get("Umpire1Name"),
        "GroundUmpire2ID":schedrecord.get("GroundUmpire2ID"),
        "Umpire2Name":summrecord.get("Umpire2Name"),
        "GroundUmpire3ID":schedrecord.get("GroundUmpire3ID"),
        "Umpire3Name":summrecord.get("Umpire3Name"),
        "RefereeID":schedrecord.get("RefereeID"),
        "Referee":summrecord.get("Referee"),
        "Scorer1Name":summrecord.get("Scorer1Name"),
        "CurrentDay":summrecord.get("CurrentDay"),
        "CurrentSession":summrecord.get("CurrentSession"),
        "PreMatchCommentary":schedrecord.get("PreMatchCommentary"),
        "PostMatchCommentary":schedrecord.get("PostMatchCommentary"),
        "TeamType":schedrecord.get("TeamType"),
        "Team1":summrecord.get("Team1"),
        "Team2":summrecord.get("Team2"),
        "PointsComments":summrecord.get("PointsComments"),
        "HomeTeamID":schedrecord.get("HomeTeamID"),
        "HomeTeamName":schedrecord.get("HomeTeamName"),
        "MatchHomeTeamLogo":schedrecord.get("MatchHomeTeamLogo"),
        "AwayTeamID":schedrecord.get("AwayTeamID"),
        "AwayTeamName":schedrecord.get("AwayTeamName"),
        "MatchAwayTeamLogo":schedrecord.get("MatchAwayTeamLogo"),
        "TossTeam":schedrecord.get("TossTeam"),
        "TossDetails":schedrecord.get("TossDetails"),
        "FirstBattingTeamID":schedrecord.get("FirstBattingTeamID"),
        "FirstBattingTeamName":schedrecord.get("FirstBattingTeamName"),
        "FirstBattingTeamCode":schedrecord.get("FirstBattingTeamCode"),
        "FirstBattingTeamLogo":summrecord.get("FirstBattingTeamLogo"),
        "FirstBattingSummary":schedrecord.get("FirstBattingSummary"),
        "Target":summrecord.get("Target"),
        "SecondBattingTeamID":schedrecord.get("SecondBattingTeamID"),
        "SecondBattingTeamName":schedrecord.get("SecondBattingTeamName"),
        "SecondBattingTeamCode":schedrecord.get("SecondBattingTeamCode"),
        "SecondBattingTeamLogo":summrecord.get("SecondBattingTeamLogo"),
        "SecondBattingSummary":schedrecord.get("SecondBattingSummary"),
        "SecondInningsFirstBattingID":schedrecord.get("SecondInningsFirstBattingID"),
        "SecondInningsFirstBattingName":schedrecord.get("SecondInningsFirstBattingName"),
        "SecondInningsSecondBattingID":schedrecord.get("SecondInningsSecondBattingID"),
        "SecondInningsSecondBattingName":schedrecord.get("SecondInningsSecondBattingName"),
        "ThirdInningsFirstBattingID":schedrecord.get("ThirdInningsFirstBattingID"),
        "ThirdInningsSecondBattingID":schedrecord.get("ThirdInningsSecondBattingID"),
        "WinningTeamID":schedrecord.get("WinningTeamID"),
        "Commentss":schedrecord.get("Commentss"),
        "1InningsName":summrecord.get("1InningsName"),
        "1Summary":summrecord.get("1Summary"),
        "1FallScore":summrecord.get("1FallScore"),
        "1FallWickets":summrecord.get("1FallWickets"),
        "1FallOvers":summrecord.get("1FallOvers"),
        "1RunRate":summrecord.get("1RunRate"),
        "Innings1Overs":summrecord.get("Innings1Overs"),
        "ProjectedScore":summrecord.get("ProjectedScore"),
        "Innings1Declare":summrecord.get("Innings1Declare"),
        "2InningsName":summrecord.get("2InningsName"),
        "2Summary":summrecord.get("2Summary"),
        "2FallScore":summrecord.get("2FallScore"),
        "2FallWickets":summrecord.get("2FallWickets"),
        "2FallOvers":summrecord.get("2FallOvers"),
        "2RunRate":summrecord.get("2RunRate"),
        "Innings2Overs":summrecord.get("Innings2Overs"),
        "2ndProjectedScore":summrecord.get("2ndProjectedScore"),
        "Innings2Declare":summrecord.get("Innings2Declare"),
        "3InningsName":summrecord.get("3InningsName"),
        "3Summary":summrecord.get("3Summary"),
        "3FallScore":summrecord.get("3FallScore"),
        "3FallWickets":summrecord.get("3FallWickets"),
        "3FallOvers":summrecord.get("3FallOvers"),
        "3RunRate":summrecord.get("3RunRate"),
        "3ndProjectedScore":summrecord.get("3ndProjectedScore"),
        "Innings3Declare":summrecord.get("Innings3Declare"),
        "4InningsName":summrecord.get("4InningsName"),
        "4Summary":summrecord.get("4Summary"),
        "4FallScore":summrecord.get("4FallScore"),
        "4FallWickets":summrecord.get("4FallWickets"),
        "4FallOvers":summrecord.get("4FallOvers"),
        "4RunRate":summrecord.get("4RunRate"),
        "Innings4Declare":summrecord.get("Innings4Declare"),
        "5InningsName":summrecord.get("5InningsName"),
        "5Summary":summrecord.get("5Summary"),
        "5FallScore":summrecord.get("5FallScore"),
        "5FallWickets":summrecord.get("5FallWickets"),
        "5FallOvers":summrecord.get("5FallOvers"),
        "5RunRate":summrecord.get("5RunRate"),
        "Innings5Declare":summrecord.get("Innings5Declare"),
        "6InningsName":summrecord.get("6InningsName"),
        "6Summary":summrecord.get("6Summary"),
        "6FallScore":summrecord.get("6FallScore"),
        "6FallWickets":summrecord.get("6FallWickets"),
        "6FallOvers":summrecord.get("6FallOvers"),
        "6RunRate":summrecord.get("6RunRate"),
        "Innings6Declare":summrecord.get("Innings6Declare"),
        "7InningsName":summrecord.get("7InningsName"),
        "7Summary":summrecord.get("7Summary"),
        "7FallScore":summrecord.get("7FallScore"),
        "7FallWickets":summrecord.get("7FallWickets"),
        "7FallOvers":summrecord.get("7FallOvers"),
        "7RunRate":summrecord.get("7RunRate"),
        "Innings7Declare":summrecord.get("Innings7Declare"),
        "8InningsName":summrecord.get("8InningsName"),
        "8Summary":summrecord.get("8Summary"),
        "8FallScore":summrecord.get("8FallScore"),
        "8FallWickets":summrecord.get("8FallWickets"),
        "8FallOvers":summrecord.get("8FallOvers"),
        "8RunRate":summrecord.get("8RunRate"),
        "Innings8Declare":summrecord.get("Innings8Declare"),
        "IsSuperOver":summrecord.get("IsSuperOver"),
        "CurrentStrikerID":schedrecord.get("CurrentStrikerID"),
        "CurrentStrikerName":summrecord.get("CurrentStrikerName"),
        "CurrentStrikerShortName":summrecord.get("CurrentStrikerShortName"),
        "StrikerImage":schedrecord.get("StrikerImage"),
        "StrikerRuns":summrecord.get("StrikerRuns"),
        "StrikerBalls":summrecord.get("StrikerBalls"),
        "StrikerFours":summrecord.get("StrikerFours"),
        "StrikerSixes":summrecord.get("StrikerSixes"),
        "StrikerSR":summrecord.get("StrikerSR"),
        "CurrentNonStrikerID":schedrecord.get("CurrentNonStrikerID"),
        "CurrentNonStrikerName":summrecord.get("CurrentNonStrikerName"),
        "CurrentNonStrikerShortName":summrecord.get("CurrentNonStrikerShortName"),
        "NonStrikerImage":schedrecord.get("NonStrikerImage"),
        "NonStrikerRuns":summrecord.get("NonStrikerRuns"),
        "NonStrikerBalls":summrecord.get("NonStrikerBalls"),
        "NonStrikerFours":summrecord.get("NonStrikerFours"),
        "NonStrikerSixes":summrecord.get("NonStrikerSixes"),
        "NonStrikerSR":summrecord.get("NonStrikerSR"),
        "CurrentBowlerID":schedrecord.get("CurrentBowlerID"),
        "CurrentBowlerName":summrecord.get("CurrentBowlerName"),
        "CurrentBowlerShortName":summrecord.get("CurrentBowlerShortName"),
        "BowlerImage":schedrecord.get("BowlerImage"),
        "BowlerOvers":summrecord.get("BowlerOvers"),
        "BowlerRuns":summrecord.get("BowlerRuns"),
        "BowlerMaidens":summrecord.get("BowlerMaidens"),
        "BowlerWickets":summrecord.get("BowlerWickets"),
        "BowlerEconomy":summrecord.get("BowlerEconomy"),
        "BowlerSR":summrecord.get("BowlerSR")
    }
    return matchSummary

#Group Standings Schema
def extract_standings(record):
    standings = {
        "StandingFlag": record.get("StandingFlag"),
        "Category": record.get("Category"),
        "CompetitionID": record.get("CompetitionID"),
        "TeamID": record.get("TeamID"),
        "TeamCode": record.get("TeamCode"),
        "TeamName": record.get("TeamName"),
        "TeamLogo": record.get("TeamLogo"),
        "Matches": record.get("Matches"),
        "Wins": record.get("Wins"),
        "Loss": record.get("Loss"),
        "Tied": record.get("Tied"),
        "NoResult": record.get("NoResult"),
        "Points": record.get("Points"),
        "Draw": record.get("Draw"),
        "ForTeams": record.get("ForTeams"),
        "AgainstTeam": record.get("AgainstTeam"),
        "NetRunRate": record.get("NetRunRate"),
        "Quotient": record.get("Quotient"),
        "OrderNo": record.get("OrderNo"),
        "IsQualified": record.get("IsQualified"),
        "LeadBy": record.get("LeadBy"),
        "Deficit": record.get("Deficit"),
        "Performance": record.get("Performance"),
        "Status": record.get("Status"),
        "MATCH_ID": record.get("MATCH_ID"),
        "PrevPosition": record.get("PrevPosition")
    }
    return standings

#Orange Cap Standings Schema
def extract_orangecap(record):
  standings = {
      "StrikerName": record.get("StrikerName"),
      "PlayerId": record.get("PlayerId"),
      "Matches": record.get("Matches"),
      "PlayerDOB": record.get("PlayerDOB"),
      "RightHandedBat": record.get("RightHandedBat"),
      "Nationality": record.get("Nationality"),
      "TCompetitionID": record.get("TCompetitionID"),
      "TStrikerID": record.get("TStrikerID"),
      "TTeamID": record.get("TTeamID"),
      "TeamCode": record.get("TeamCode"),
      "TeamName": record.get("TeamName"),
      "CompetitionID": record.get("CompetitionID"),
      "TeamID": record.get("TeamID"),
      "StrikerID": record.get("StrikerID"),
      "Innings": record.get("Innings"),
      "Extras": record.get("Extras"),
      "TotalRuns": record.get("TotalRuns"),
      "Balls": record.get("Balls"),
      "Dotballs": record.get("Dotballs"),
      "StrikeRate": record.get("StrikeRate"),
      "DBPercent": record.get("DBPercent"),
      "DBFreq": record.get("DBFreq"),
      "BdryFreq": record.get("BdryFreq"),
      "BdryPercent": record.get("BdryPercent"),
      "RPSS": record.get("RPSS"),
      "ScoringBalls": record.get("ScoringBalls"),
      "Ones": record.get("Ones"),
      "Twos": record.get("Twos"),
      "Threes": record.get("Threes"),
      "Fours": record.get("Fours"),
      "Sixes": record.get("Sixes"),
      "Outs": record.get("Outs"),
      "NotOuts": record.get("NotOuts"),
      "BattingAveragesss": record.get("BattingAveragesss"),
      "FiftyPlusRuns": record.get("FiftyPlusRuns"),
      "Centuries": record.get("Centuries"),
      "DoubleCenturies": record.get("DoubleCenturies"),
      "HighestScore": record.get("HighestScore"),
      "BattingAverage": record.get("BattingAverage"),
      "Catches": record.get("Catches"),
      "Stumpings": record.get("Stumpings"),
      "ClientPlayerID": record.get("ClientPlayerID")
  }
  return standings

#Purple Cap Standings Schema
def extract_purplecap(record):
  standings = {
      "BowlerName": record.get("BowlerName"),
      "RightHandedBat": record.get("RightHandedBat"),
      "Nationality": record.get("Nationality"),
      "TeamCode": record.get("TeamCode"),
      "TeamName": record.get("TeamName"),
      "Matches": record.get("Matches"),
      "CompetitionID": record.get("CompetitionID"),
      "Innings": record.get("Innings"),
      "TeamID": record.get("TeamID"),
      "BowlerID": record.get("BowlerID"),
      "LegalBallsBowled": record.get("LegalBallsBowled"),
      "TotalRunsConceded": record.get("TotalRunsConceded"),
      "DotBallsBowled": record.get("DotBallsBowled"),
      "DotBallPercent": record.get("DotBallPercent"),
      "ScoringBallsBowled": record.get("ScoringBallsBowled"),
      "BowlingAverage": record.get("BowlingAverage"),
      "StrikeRate": record.get("StrikeRate"),
      "BowlingSR": record.get("BowlingSR"),
      "BoundaryPercentage": record.get("BoundaryPercentage"),
      "BoundaryFrequency": record.get("BoundaryFrequency"),
      "EconomyRate": record.get("EconomyRate"),
      "OversBowled": record.get("OversBowled"),
      "Ones": record.get("Ones"),
      "Twos": record.get("Twos"),
      "Threes": record.get("Threes"),
      "Fours": record.get("Fours"),
      "Sixes": record.get("Sixes"),
      "Wides": record.get("Wides"),
      "NoBalls": record.get("NoBalls"),
      "Byes": record.get("Byes"),
      "LegBye": record.get("LegBye"),
      "Wickets": record.get("Wickets"),
      "InningsRuns": record.get("InningsRuns"),
      "InningsWickets": record.get("InningsWickets"),
      "MatchRuns": record.get("MatchRuns"),
      "MatchWickets": record.get("MatchWickets"),
      "BBIW": record.get("BBIW"),
      "BBMW": record.get("BBMW"),
      "Maidens": record.get("Maidens"),
      "MaidenWickets": record.get("MaidenWickets"),
      "FourWickets": record.get("FourWickets"),
      "FiveWickets": record.get("FiveWickets"),
      "TenWickets": record.get("TenWickets"),
      "ClientPlayerID": record.get("ClientPlayerID")
  }
  return standings

#Matches
# Batting Score Card Schema
def extract_battingCard(record):

    battingCard = {
        "MatchID": record.get("MatchID"),
        "InningsNo": record.get("InningsNo"),
        "TeamID": record.get("TeamID"),
        "PlayerID": record.get("PlayerID"),
        "PLAYER_ID": record.get("PLAYER_ID"),
        "PlayerName": record.get("PlayerName"),
        "PlayingOrder": record.get("PlayingOrder"),
        "MatchPlayingOrder": record.get("MatchPlayingOrder"),
        "BowlerName": record.get("BowlerName"),
        "OutDesc": record.get("OutDesc"),
        "Runs": record.get("Runs"),
        "Balls": record.get("Balls"),
        "DotBalls": record.get("DotBalls"),
        "DotBallPercentage": record.get("DotBallPercentage"),
        "DotBallFrequency": record.get("DotBallFrequency"),
        "Ones": record.get("Ones"),
        "Twos": record.get("Twos"),
        "Threes": record.get("Threes"),
        "Fours": record.get("Fours"),
        "Sixes": record.get("Sixes"),
        "BoundaryPercentage": record.get("BoundaryPercentage"),
        "BoundaryFrequency": record.get("BoundaryFrequency"),
        "StrikeRate": record.get("StrikeRate"),
        "MinOver": record.get("MinOver"),
        "MinStrikerOver": record.get("MinStrikerOver"),
        "WicketNo": record.get("WicketNo"),
        "AgainstFast": record.get("AgainstFast"),
        "AgainstSpin": record.get("AgainstSpin"),
        "AgainstFastPercent": record.get("AgainstFastPercent"),
        "AgainstSpinPercent": record.get("AgainstSpinPercent")
    }

    return battingCard

# Bowling Score Card Schema
def extract_bowlingCard(record):

    bowlingCard = {
        "MatchID": record.get("MatchID"),
        "InningsNo": record.get("InningsNo"),
        "TeamID": record.get("TeamID"),
        "PlayerID": record.get("PlayerID"),
        "PlayerName": record.get("PlayerName"),
        "Overs": record.get("Overs"),
        "Maidens": record.get("Maidens"),
        "Runs": record.get("Runs"),
        "Wickets": record.get("Wickets"),
        "Wides": record.get("Wides"),
        "NoBalls": record.get("NoBalls"),
        "Economy": record.get("Economy"),
        "BowlingOrder": record.get("BowlingOrder"),
        "TotalLegalBallsBowled": record.get("TotalLegalBallsBowled"),
        "ScoringBalls": record.get("ScoringBalls"),
        "DotBalls": record.get("DotBalls"),
        "DBPercent": record.get("DBPercent"),
        "DBFrequency": record.get("DBFrequency"),
        "Ones": record.get("Ones"),
        "Twos": record.get("Twos"),
        "Threes": record.get("Threes"),
        "Fours": record.get("Fours"),
        "Sixes": record.get("Sixes"),
        "BdryPercent": record.get("BdryPercent"),
        "BdryFreq": record.get("BdryFreq"),
        "WicketTakingRate": record.get("StrikeRate"),
        "FourPercent": record.get("FourPercent"),
        "SixPercent": record.get("SixPercent")
    }

    return bowlingCard

# Batting Head To Head Schema
def extract_battingheadtohead(match_id,record):
    battingheadtohead = {
        "MatchID": str(match_id),
        "BatsManID": record.get("BatsManID"),
        "BowlerID": record.get("BowlerID"),
        "BatsManName": record.get("BatsManName"),
        "BowlerName": record.get("BowlerName"),
        "Runs": record.get("Runs"),
        "RunsThroughExtras": record.get("RunsThroughExtras"),
        "TotalRuns": record.get("TotalRuns"),
        "BallsFaced": record.get("BallsFaced"),
        "StrikeRate": record.get("StrikeRate"),
        "DotBalls": record.get("DotBalls"),
        "DBPercent": record.get("DBPercent"),
        "DotBallFrequency": record.get("DotBallFrequency"),
        "ScoringBalls": record.get("ScoringBalls"),
        "Ones": record.get("Ones"),
        "Twos": record.get("Twos"),
        "Threes": record.get("Threes"),
        "Fours": record.get("Fours"),
        "Sixes": record.get("Sixes"),
        "BdryFreq": record.get("BdryFreq"),
        "BdryPercent": record.get("BdryPercent"),
        "WicketLost": record.get("WicketLost"),
        "RPSS": record.get("RPSS")
    }
    return battingheadtohead

# Bowling Head to Head Schema
def extract_bowlingheadtohead(record):
    bowlingheadtohead = {
        "MatchID": record.get("MatchID"),
        "BowlerID": record.get("BowlerID"),
        "BowlerName": record.get("BowlerName"),
        "StrikerID": record.get("StrikerID"),
        "BatsManName": record.get("BatsManName"),
        "TotalRunsConceded": record.get("TotalRunsConceded"),
        "BowlingAverage": record.get("BowlingAverage"),
        "TotalLegalBallsBowled": record.get("TotalLegalBallsBowled"),
        "BowlingSR": record.get("BowlingSR"),
        "EconomyRate": record.get("EconomyRate"),
        "DotBallsBowled": record.get("DotBallsBowled"),
        "ScoringBalls": record.get("ScoringBalls"),
        "Ones": record.get("Ones"),
        "Twoes": record.get("Twoes"),
        "Threes": record.get("Threes"),
        "Bdry4": record.get("Bdry4"),
        "Bdry6": record.get("Bdry6"),
        "Wides": record.get("Wides"),
        "NoBalls": record.get("NoBalls"),
        "Wickets": record.get("Wickets"),
        "RunOutsDone": record.get("RunOutsDone"),
        "BowledOutsDone": record.get("BowledOutsDone"),
        "StumpingsDone": record.get("StumpingsDone"),
        "BowlingDotBallPercentage": record.get("BowlingDotBallPercentage"),
        "BowlingDotBallFrequency": record.get("BowlingDotBallFrequency"),
        "BowlingBdryPercentage": record.get("BowlingBdryPercentage"),
        "BowlingBdryFrequency": record.get("BowlingBdryFrequency")
    }
    return bowlingheadtohead

# Extras Schema
def extract_extras(records):
    extras = {
        "MatchID": records.get("MatchID"),
        "InningsNo": records.get("InningsNo"),
        "TeamID": records.get("TeamID"),
        "Total": records.get("Total"),
        "TotalExtras": records.get("TotalExtras"),
        "Byes": records.get("Byes"),
        "LegByes": records.get("LegByes"),
        "NoBalls": records.get("NoBalls"),
        "Wides": records.get("Wides"),
        "Penalty": records.get("Penalty"),
        "CurrentRunRate": records.get("CurrentRunRate"),
        "FallScore": records.get("FallScore"),
        "FallWickets": records.get("FallWickets"),
        "FallOvers": records.get("FallOvers"),
        "BattingTeamName": records.get("BattingTeamName"),
        "BowlingTeamName": records.get("BowlingTeamName"),
        "MaxPartnerShipRuns": records.get("MaxPartnerShipRuns"),
    }
    return extras

#Partnership Break Schema
def extract_PartnershipBreak(match_id,record):
    PartnershipBreak = {
        "MatchID": str(match_id),
        "InningsNo": record.get("InningsNo"),
        "OverNo": record.get("OverNo"),
        "WicketNo": record.get("WicketNo"),
        "WicketType": record.get("WicketType"),
        "OutBatsmanID": record.get("OutBatsmanID")
    }
    return PartnershipBreak

# Partnership Scores Schema
def extract_PartnershipScores(record):
    PartnershipScores = {
        "MatchID": record.get("MatchID"),
        "BattingTeamID": record.get("BattingTeamID"),
        "InningsNo": record.get("InningsNo"),
        "StrikerID": record.get("StrikerID"),
        "Striker": record.get("Striker"),
        "NonStrikerID": record.get("NonStrikerID"),
        "NonStriker": record.get("NonStriker"),
        "PartnershipTotal": record.get("PartnershipTotal"),
        "StrikerRuns": record.get("StrikerRuns"),
        "StrikerBalls": record.get("StrikerBalls"),
        "Extras": record.get("Extras"),
        "NonStrikerRuns": record.get("NonStrikerRuns"),
        "NonStrikerBalls": record.get("NonStrikerBalls"),
        "MatchMaxOver": record.get("MatchMaxOver"),
        "MatchMinOver": record.get("MatchMinOver"),
        "I_0": record.get("I_0"),
        "RowNumber": record.get("RowNumber")
    }
    return PartnershipScores

# Manhattan Graph Schema
def extract_ManhattanGraph(match_id,record):
    ManhattanGraph = {
        "MatchID": str(match_id),
        "InningsNo": record.get("InningsNo"),
        "OverNo": record.get("OverNo"),
        "BattingTeamID": record.get("BattingTeamID"),
        "OverRuns": record.get("OverRuns"),
        "BowlerRuns": record.get("BowlerRuns"),
        "BowlerID": record.get("BowlerID"),
        "Wickets": record.get("Wickets"),
        "Bowler": record.get("Bowler")
    }
    return ManhattanGraph

# Manhattan Wicket Schema
def extract_ManhattanWickets(match_id,record):
    ManhattanWickets = {
        "MatchID": str(match_id),
        "InningsNo": record.get("InningsNo"),
        "OverNo": record.get("OverNo"),
        "BattingTeamID": record.get("BattingTeamID"),
        "OutBatsman": record.get("OutBatsman"),
        "OutDesc": record.get("OutDesc"),
        "BatsmanRuns": record.get("BatsmanRuns"),
        "BatsmanBalls": record.get("BatsmanBalls"),
        "OutBatsMan2": record.get("OutBatsMan2"),
        "BatsManRuns2": record.get("BatsManRuns2")
    }
    return ManhattanWickets

# Fall of Wickets Schema
def extract_FallOfWickets(record):
    FallOfWickets = {
        "MatchID": record.get("MatchID"),
        "InningsNo": record.get("InningsNo"),
        "TeamID": record.get("TeamID"),
        "PlayerID": record.get("PlayerID"),
        "PlayerName": record.get("PlayerName"),
        "FallScore": record.get("FallScore"),
        "FallWickets": record.get("FallWickets"),
        "FallOvers": record.get("FallOvers")
    }
    return FallOfWickets

# Wagon Wheel Schema
def extract_WagonWheel(match_id,record):
    WagonWheel = {
        "MatchID": str(match_id),
        "BallID": record.get("BallID"),
        "StrikerID": record.get("StrikerID"),
        "BowlerID": record.get("BowlerID"),
        "FielderAngle": record.get("FielderAngle"),
        "FielderLengthRatio": record.get("FielderLengthRatio"),
        "Runs": record.get("Runs"),
        "IsFour": record.get("IsFour"),
        "IsSix": record.get("IsSix"),
        "BatType": record.get("BatType")
    }
    return WagonWheel

# Wagon Wheel Summary Schema
def extract_WagonWheelSummary(record):
    WagonWheelSummary = {
        "MatchID": record.get("MatchID"),
        "BattingTeamID": record.get("BattingTeamID"),
        "InningsNo": record.get("InningsNo"),
        "Ones": record.get("Ones"),
        "Twos": record.get("Twos"),
        "Threes": record.get("Threes"),
        "Fours": record.get("Fours"),
        "Sixes": record.get("Sixes")
    }
    return WagonWheelSummary

# Over History Schema
def extract_OverHistory(record):
    OverHistory = {
        "BallID": record.get("BallID"),
        "BallUniqueID": record.get("BallUniqueID"),
        "MatchID": record.get("MatchID"),
        "InningsNo": record.get("InningsNo"),
        "OverNo": record.get("OverNo"),
        "BallNo": record.get("BallNo"),
        "ActualBallNo": record.get("ActualBallNo"),
        "BattingTeamID": record.get("BattingTeamID"),
        "TeamName": record.get("TeamName"),
        "StrikerID": record.get("StrikerID"),
        "NonStrikerID": record.get("NonStrikerID"),
        "BatsManName": record.get("BatsManName"),
        "BowlerID": record.get("BowlerID"),
        "BowlerName": record.get("BowlerName"),
        "Runs": record.get("BallRuns"),
        "ActualRuns": record.get("ActualRuns"),
        "IsOne": record.get("IsOne"),
        "IsTwo": record.get("IsTwo"),
        "IsThree": record.get("IsThree"),
        "IsDotball": record.get("IsDotball"),
        "Extras": record.get("Extras"),
        "IsWide": record.get("IsWide"),
        "IsNoBall": record.get("IsNoBall"),
        "IsBye": record.get("IsBye"),
        "IsLegBye": record.get("IsLegBye"),
        "IsFour": record.get("IsFour"),
        "IsSix": record.get("IsSix"),
        "IsWicket": record.get("IsWicket"),
        "WicketType": record.get("WicketType"),
        "Wickets": record.get("Wickets"),
        "IsBowlerWicket": record.get("IsBowlerWicket"),
        "CommentStrikers": record.get("CommentStrikers"),
        "NewCommentry": record.get("NewCommentry"),
        "Commentry": record.get("Commentry"),
        "UPDCommentry": record.get("UPDCommentry"),
        "Day": record.get("Day"),
        "SESSION_NO": record.get("SESSION_NO"),
        "IsExtra": record.get("IsExtra"),
        "OutBatsManID": record.get("OutBatsManID"),
        "SNO": record.get("SNO"),
        "Xpitch": record.get("Xpitch"),
        "Ypitch": record.get("Ypitch"),
        "RunRuns": record.get("RunRuns"),
        "IsMaiden": record.get("IsMaiden"),
        "OverImage": record.get("OverImage"),
        "BowlTypeID": record.get("BowlTypeID"),
        "BowlTypeName": record.get("BowlTypeName"),
        "ShotTypeID": record.get("ShotTypeID"),
        "ShotType": record.get("ShotType"),
        "IsBouncer": record.get("IsBouncer"),
        "IsFreeHit": record.get("IsFreeHit"),
        "BallCount": record.get("BallCount"),
        "BCCheck": record.get("BCCheck"),
        "TotalRuns": record.get("TotalRuns"),
        "TotalWickets": record.get("TotalWickets"),
        "BOWLING_LINE_ID": record.get("BOWLING_LINE_ID"),
        "BOWLING_LENGTH_ID": record.get("BOWLING_LENGTH_ID"),
        "FiveHaul": record.get("FiveHaul"),
        "HatCheck": record.get("HatCheck"),
        "Flag": record.get("Flag"),
        "FlagSet": record.get("FlagSet"),
        "PenaltyRuns": record.get("PenaltyRuns"),
        "IsFifty": record.get("IsFifty"),
        "IsHundred": record.get("IsHundred"),
        "IsTwoHundred": record.get("IsTwoHundred"),
        "IsHattrick": record.get("IsHattrick"),
        "STclientID": record.get("STclientID"),
        "NSTclientID": record.get("NSTclientID")
    }
    return OverHistory

# Squad Schema
def extract_squad(matchID, record):
    squad = {
        "MatchID": matchID,
        "PlayerID": record.get("PlayerID"),
        "TeamID": record.get("TeamID"),
        "TeamCode": record.get("TeamCode"),
        "TeamName": record.get("TeamName"),
        "PlayerName": record.get("PlayerName"),
        "PlayerShortName": record.get("PlayerShortName"),
        "BattingType": record.get("BattingType"),
        "BowlingProficiency": record.get("BowlingProficiency"),
        "PlayerSkill": record.get("PlayerSkill"),
        "IsCaptain": record.get("IsCaptain"),
        "IsViceCaptain": record.get("IsViceCaptain"),
        "IsWK": record.get("IsWK"),
        "IsNonDomestic": record.get("IsNonDomestic"),
        "ClientPlayerID": record.get("ClientPlayerID"),
        "PlayingOrder": record.get("PlayingOrder")
    }
    return squad

#Teams Head to Head Schema
def extract_team_h2h(record):
    team_h2h = {
        "Team1ID": record.get("Team1ID"),
        "Team2ID": record.get("Team2ID"),
        "Team1Name": record.get("Team1Name"),
        "Team2Name": record.get("Team2Name"),
        "TotalMatches": record.get("TotalMatches"),
        "Team1Won": record.get("Team1Won"),
        "Team2Won": record.get("Team2Won"),
        "NoResult": record.get("NoResult")
    }
    return team_h2h


## API's

In [ ]:
years_url = "https://ipl-stats-sports-mechanic.s3.ap-south-1.amazonaws.com/ipl/mc/competition.js"
auction_url = "https://www.iplt20.com/auction/"
playersCarrier_url = "https://ipl-stats-sports-mechanic.s3.ap-south-1.amazonaws.com/ipl/feeds/stats/player/allPlayerCarrierStats.js"
base_schedule_url = "https://ipl-stats-sports-mechanic.s3.ap-south-1.amazonaws.com/ipl/feeds/{year}-matchschedule.js"
base_summary_url = "https://ipl-stats-sports-mechanic.s3.ap-south-1.amazonaws.com/ipl/feeds/{MatchID}-matchsummary.js"
groupstandings_url = "https://ipl-stats-sports-mechanic.s3.ap-south-1.amazonaws.com/ipl/feeds/stats/{year}-groupstandings.js"
orangecap_standings_url = "https://ipl-stats-sports-mechanic.s3.ap-south-1.amazonaws.com/ipl/feeds/stats/{year}-toprunsscorers.js"
purplecap_standings_url = "https://ipl-stats-sports-mechanic.s3.ap-south-1.amazonaws.com/ipl/feeds/stats/{year}-mostwickets.js"
match_squad_url = "https://ipl-stats-sports-mechanic.s3.ap-south-1.amazonaws.com/ipl/feeds/{MatchID}-squad.js?callback=onsquad&_=1756978713167"
base_innings_url_1 = "https://ipl-stats-sports-mechanic.s3.ap-south-1.amazonaws.com/ipl/feeds/{MatchID}-Innings1.js"
base_innings_url_2 = "https://ipl-stats-sports-mechanic.s3.ap-south-1.amazonaws.com/ipl/feeds/{MatchID}-Innings2.js"
team_h2h_url = "https://ipl-stats-sports-mechanic.s3.ap-south-1.amazonaws.com/ipl/feeds/stats/match/{MatchID}-headTohead.js"

##Resources

In [ ]:
YearsList = []
SeasonList = []
EventList = []
PlayerBattingCarrier = []
PlayerBowlingCarrier = []
MatchSummary = []
GroupStandings = []
OrangeCap = []
PurpleCap = []
BattingCard = []
BowlingCard = []
Extras = []
FallOfWickets = []
WagonWheel = []
PartnershipScores = []
PartnershipBreak = []
ManhattanGraph = []
ManhattanWickets = []
OverHistory = []
WagonWheelSummary = []
Battingheadtohead = []
Bowlingheadtohead = []
Squad = []
Team_h2h =[]

## Fetch Seasons

In [ ]:
data = yearslist(years_url)

for id in data['competition']:
  YearsList.append(id['CompetitionID'])
  SeasonList.append(int(id['CompetitionName'].strip('IPL ')))

for season in SeasonList:
  print(season,end=', ')

2026, 2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016, 2015, 2014, 2013, 2012, 2011, 2010, 2009, 2008, 

##Fetch Entire Seasons Data

###Auction Data

####Auction Stats Details

In [ ]:
records = []

for season in SeasonList:
    print(f"Fetching IPL Season {season} Stats...")

    url = auction_url + str(season)
    response = fetch_webpages(url)
    soup = BeautifulSoup(response.content, "lxml")

    row = {}
    row["season"] = season

    # -----------------------
    # Auction Highlight
    # -----------------------
    auction_highlight_element = soup.find("div", class_="modal-body")

    if auction_highlight_element:
        raw = [i.strip() for i in auction_highlight_element.text.split("\n") if i.strip()]
        row["auction_highlight"] = " ".join(raw)
    else:
        row["auction_highlight"] = "N/A"

    # -----------------------
    # Most Expensive Player
    # -----------------------
    mep_data_element = soup.find("div", class_="auction-most-exp-style")

    if mep_data_element:
        raw = [i.strip() for i in mep_data_element.text.split("\n") if i.strip()]

        for i in range(0, len(raw)-1, 2):
            key = raw[i].replace("₹", "INR")
            value = raw[i+1]
            row[key] = value
    else:
        row["Most Expensive Player"] = "N/A"

    # -----------------------
    # Auction Stats
    # -----------------------
    auction_stats_data_element = soup.find("div", class_="auction-header__stats")

    if auction_stats_data_element:
        cleaned_lines = [i.strip() for i in auction_stats_data_element.text.split("\n") if i.strip()]

        for i in range(0, len(cleaned_lines), 2):
            if i + 1 < len(cleaned_lines):
                value = cleaned_lines[i]
                key = cleaned_lines[i+1]
                row[key] = value
    else:
        row["Players sold"] = "N/A"
        row["Overseas Players"] = "N/A"
        row["Total Spent"] = "N/A"

    records.append(row)

df_auction_stats = pd.DataFrame(records)
print("✅ Successfully Fetched Auction Stats")

Fetching IPL Season 2026 Stats...
Successfully fetched the webpage
Fetching IPL Season 2025 Stats...
Successfully fetched the webpage
Fetching IPL Season 2024 Stats...
Successfully fetched the webpage
Fetching IPL Season 2023 Stats...
Successfully fetched the webpage
Fetching IPL Season 2022 Stats...
Successfully fetched the webpage
Fetching IPL Season 2021 Stats...
Successfully fetched the webpage
Fetching IPL Season 2020 Stats...
Successfully fetched the webpage
Fetching IPL Season 2019 Stats...
Successfully fetched the webpage
Fetching IPL Season 2018 Stats...
Successfully fetched the webpage
Fetching IPL Season 2017 Stats...
Successfully fetched the webpage
Fetching IPL Season 2016 Stats...
Successfully fetched the webpage
Fetching IPL Season 2015 Stats...
Successfully fetched the webpage
Fetching IPL Season 2014 Stats...
Successfully fetched the webpage
Fetching IPL Season 2013 Stats...
Successfully fetched the webpage
Fetching IPL Season 2012 Stats...
Successfully fetched the web

####Auction Team Stats

In [ ]:
all_rows = []

for season in SeasonList:

    print(f"Fetching IPL Season {season} Stats...")

    url = auction_url + str(season)
    response = fetch_webpages(url)
    soup = BeautifulSoup(response.content, "lxml")

    # First, find the main container div (BeautifulSoup object)
    auction_container = soup.find("div", class_="auction-grid-view")

    if not auction_container:
        print(f"Error: The 'auction-grid-view' container div was not found for season {season}. Skipping...")
        continue # Skip to the next season if the main container is not found

    # Extract team div elements directly, as each contains a team's stats
    team_div_elements = auction_container.find_all(lambda tag: tag.name == "div" and tag.has_attr("id"))

    if not team_div_elements:
        print(f"Warning: No team data found within 'auction-grid-view' for season {season}. Skipping...")
        continue

    for team_div in team_div_elements:
        team_code = team_div["id"]
        team_stats_text = team_div.text

        # Clean the text for this specific team
        cleaned_stats = [item.strip() for item in team_stats_text.split('\n') if item.strip()]

        team_stats_dict = {}
        # Parse key-value pairs using a more explicit search
        for i in range(len(cleaned_stats)):
            if cleaned_stats[i] == 'Funds Remaining' and i + 1 < len(cleaned_stats):
                team_stats_dict['Funds Remaining'] = cleaned_stats[i+1]
            elif cleaned_stats[i] == 'Overseas Players' and i + 1 < len(cleaned_stats):
                team_stats_dict['Overseas Players'] = cleaned_stats[i+1]
            elif cleaned_stats[i] == 'Total Players' and i + 1 < len(cleaned_stats):
                team_stats_dict['Total Players'] = cleaned_stats[i+1]

        # Extract values safely using .get() with default empty string
        funds_remaining_str = team_stats_dict.get("Funds Remaining", "0").replace("₹", "").replace(",", "").strip()
        overseas_players_str = team_stats_dict.get("Overseas Players", "0").strip()
        total_players_str = team_stats_dict.get("Total Players", "0").strip()

        # Convert to int, handling potential non-digit values safely
        funds_remaining = int(funds_remaining_str) if funds_remaining_str.isdigit() else 0
        overseas_player_count = int(overseas_players_str) if overseas_players_str.isdigit() else 0
        total_player_count = int(total_players_str) if total_players_str.isdigit() else 0

        all_rows.append({
            "season": season,
            "team_code": team_code,
            "funds_remaining": funds_remaining,
            "overseas_player_count": overseas_player_count,
            "total_player_count": total_player_count
        })

df_auction_team_stats = pd.DataFrame(all_rows)

print("✅ Successfully Fetched Auction Teams Stats")

Fetching IPL Season 2026 Stats...
Successfully fetched the webpage
Fetching IPL Season 2025 Stats...
Successfully fetched the webpage
Fetching IPL Season 2024 Stats...
Successfully fetched the webpage
Fetching IPL Season 2023 Stats...
Successfully fetched the webpage
Fetching IPL Season 2022 Stats...
Successfully fetched the webpage
Error: The 'auction-grid-view' container div was not found for season 2022. Skipping...
Fetching IPL Season 2021 Stats...
Successfully fetched the webpage
Error: The 'auction-grid-view' container div was not found for season 2021. Skipping...
Fetching IPL Season 2020 Stats...
Successfully fetched the webpage
Error: The 'auction-grid-view' container div was not found for season 2020. Skipping...
Fetching IPL Season 2019 Stats...
Successfully fetched the webpage
Error: The 'auction-grid-view' container div was not found for season 2019. Skipping...
Fetching IPL Season 2018 Stats...
Successfully fetched the webpage
Error: The 'auction-grid-view' container div 

####Auction Players List Details

In [ ]:
all_rows = []

for season in SeasonList:

  print(f"Fetching IPL Season {season} Stats...")

  url = auction_url + str(season)
  response = fetch_webpages(url)
  soup = BeautifulSoup(response.content, "lxml")

  top_buys = soup.find("div", id="top-buys")
  if not top_buys:
    print(f"Warning: No top-buys data found for season {season}. Skipping...")
  else:
    top_buys_data = top_buys.text.replace('\n','/')
    top_buys_data = re.sub(r'/+', '|', top_buys_data).split('|')

  top_buys_stage = []
  for item in top_buys_data:
    if item.strip().lower() == 'sr. no.':
      continue
    elif item.strip():
      top_buys_stage.append(item.replace(',','').replace('₹','').strip())

  remaining_players = soup.find("div", id="autab4-2022")
  if not remaining_players:
    print(f"Warning: No remaining-players data found for season {season}. Skipping...")
  else:
    remaining_players_data = remaining_players.text.replace('\n','&')
    remaining_players_data = re.sub(r'&+', '|', remaining_players_data).split('|')
  remaining_players_stage = []
  for item in remaining_players_data:
    if item.strip().lower() == 'sr. no.':
      continue
    elif item.strip():
      remaining_players_stage.append(item.replace(',','').replace('₹','').strip())

  # print(remaining_players_stage)

  final = []

  # --- TOP BUYS ---
  i = 4  # skip header
  while i < len(top_buys_stage)-4:
    if season not in (2026,2025):
      player_team = top_buys_stage[i]
      player = top_buys_stage[i+1]
      player_type = top_buys_stage[i+2]
      player_base_price = 0
      player_winning_bid = top_buys_stage[i+3]

      obj = {
        "Team": player_team,
        "Player": player,
        "Nationality": None,
        "Player_Type": player_type,
        "Base_Price": player_base_price,
        "Winning_Bid": player_winning_bid,
        "Capped/Uncapped": None,
        "is_top_buys": True
      }

      final.append(obj)
      i += 4
    else:
      player_team = ''
      player = top_buys_stage[i]
      player_type = ''
      player_base_price = top_buys_stage[i+1]
      player_winning_bid = top_buys_stage[i+2]

      obj = {
        "Team": player_team,
        "Player": player,
        "Nationality": None,
        "Player_Type": player_type,
        "Base_Price": player_base_price,
        "Winning_Bid": player_winning_bid,
        "Capped/Uncapped": None,
        "is_top_buys": True
      }

      final.append(obj)
      i += 3

  # --- REMAINING PLAYERS ---

  if season not in (2026,2025):
    i = 4  # skip header
  else:
    i = 3  # skip header

  while i < len(remaining_players_stage)-4:
    if season not in (2026,2025):
      player = remaining_players_stage[i]
      player_nationality = remaining_players_stage[i+1]
      player_type = remaining_players_stage[i+2]
      player_base_price = remaining_players_stage[i+3]
      player_experience_type = ''

      obj = {
        "Team": None,
        "Player": player,
        "Nationality": player_nationality,
        "Player_Type": player_type,
        "Base_Price": player_base_price,
        "Winning_Bid": None,
        "Capped/Uncapped": player_experience_type,
        "is_top_buys": False
      }

      final.append(obj)

      i += 4
    else:
      player = remaining_players_stage[i]
      player_nationality = ''
      player_type = ''
      player_base_price = remaining_players_stage[i+1]
      player_experience_type = remaining_players_stage[i+2]

      obj = {
        "Team": None,
        "Player": player,
        "Nationality": player_nationality,
        "Player_Type": player_type,
        "Base_Price": player_base_price,
        "Winning_Bid": None,
        "Capped/Uncapped": player_experience_type,
        "is_top_buys": False
      }

      final.append(obj)

      i += 3

  # Append all collected players for the current season to all_rows
  for p_obj in final:
      all_rows.append({
          "Season": season,
          "Team": p_obj['Team'],
          "Player": p_obj['Player'],
          "Nationality": p_obj['Nationality'],
          "Player Type": p_obj['Player_Type'],
          "Base Price": p_obj['Base_Price'],
          "Winning Bid": p_obj['Winning_Bid'],
          "Capped/Uncapped": p_obj['Capped/Uncapped'],
          "is_top_buys": p_obj['is_top_buys']
      })

# Create DataFrame after processing all seasons
df_player_auction_list = pd.DataFrame(all_rows)

df_player_auction_list['Base Price'] = pd.to_numeric(df_player_auction_list['Base Price'], errors='coerce')
df_player_auction_list['Winning Bid'] = pd.to_numeric(df_player_auction_list['Winning Bid'], errors='coerce')

print("✅ Successfully Fetched Auction Players list Details")

Fetching IPL Season 2026 Stats...
Successfully fetched the webpage
Fetching IPL Season 2025 Stats...
Successfully fetched the webpage
Fetching IPL Season 2024 Stats...
Successfully fetched the webpage
Fetching IPL Season 2023 Stats...
Successfully fetched the webpage
Fetching IPL Season 2022 Stats...
Successfully fetched the webpage
Fetching IPL Season 2021 Stats...
Successfully fetched the webpage
Fetching IPL Season 2020 Stats...
Successfully fetched the webpage
Fetching IPL Season 2019 Stats...
Successfully fetched the webpage
Fetching IPL Season 2018 Stats...
Successfully fetched the webpage
Fetching IPL Season 2017 Stats...
Successfully fetched the webpage
Fetching IPL Season 2016 Stats...
Successfully fetched the webpage
Fetching IPL Season 2015 Stats...
Successfully fetched the webpage
Fetching IPL Season 2014 Stats...
Successfully fetched the webpage
Fetching IPL Season 2013 Stats...
Successfully fetched the webpage
Fetching IPL Season 2012 Stats...
Successfully fetched the web

####Auction Sold Players Details

In [ ]:
all_rows = []

for season in SeasonList:

  print(f"Fetching IPL Season {season} Stats...")

  url = auction_url + str(season)
  response = fetch_webpages(url)
  soup = BeautifulSoup(response.content, "lxml")

  auction_rec = soup.find("div", class_="tab-pane")
  if not auction_rec:
    print(f"Warning: No auction-record data found for season {season}. Skipping...")
    continue
  else:
    # Collect all text from sections into a flat list, splitting by '&+' delimiter
    processed_stats_text = []
    auction_rec_sections = auction_rec.find_all("section", class_="ih-points-table-sec")
    for section_item in auction_rec_sections:
      cleaned_section = section_item.text.replace("\n","&")
      processed_stats_text.extend([s.strip() for s in re.split(r'&+', cleaned_section) if s.strip()])

  final = [item.replace(',','').strip() for item in processed_stats_text if item.strip()]

  teams = {} # Reset teams for each season
  current_team = None # Track the current team being processed
  i = 0

  # Headers used to distinguish team names from player data. Team name should not be in this set.
  headers_for_detection = {"Sr. No.", "Player","Nationality","Player_Type", "Base Price", "Winning Bid", "Capped/Uncapped"}

  while i < len(final):
    # Check if the current item is a team name header followed by 'sr. no.'
    if i + 1 < len(final) and final[i] not in headers_for_detection and final[i+1].lower() == 'sr. no.':
      current_team = final[i]
      teams[current_team] = [] # Initialize the list for the new team directly
      # print(f"Detected Team: {current_team} for season {season}") # Debugging
      i += 6 # Skip the team name and the 5 header columns ('SR. NO.', 'PLAYER', etc.)
      continue # Move to the next iteration to process players for this new team

    # If a current_team is set, attempt to parse a player record
    if current_team and current_team in teams:
      if season in (2026,2025): # Newer seasons have different player record format
        if i + 4 < len(final): # Ensure there are enough elements for a full player record (5 items)
          try:
            player_record = {
              "Sr No": int(final[i]),
              "Player": final[i+1],
              "Nationality": None,
              "Player_Type": None,
              "Base Price": int(final[i+2]),
              "Winning Bid": int(final[i+3]),
              "Status": final[i+4]
            }
            teams[current_team].append(player_record)
            i += 5 # Move past this player's 5 data points
            continue # Continue to next iteration, expecting another player or new team
          except ValueError: # Catch conversion errors if it's not a player record
            pass # Fall through to general increment if not a player record
      else: # Older seasons have a different player record format
        if i + 4 < len(final): # Ensure there are enough elements for a full player record (5 items)
          try:
            player_record = {
              "Sr. No.": final[i],
              "Player": final[i+1],
              "Nationality": final[i+2],
              "Player_Type": final[i+3],
              "Base Price": None, # Not explicitly available in this format from extraction
              "Winning Bid": final[i+4],
              "Status": None # Not explicitly available
            }
            teams[current_team].append(player_record)
            i += 5 # Move past this player's 5 data points
            continue # Continue to next iteration
          except Exception: # Catch any other parsing errors
            pass # Fall through to general increment

    i += 1 # Default increment for 'i' if no specific condition was met or player record was skipped

  # After processing all raw items for the current season, populate all_rows
  for team_name, players_list in teams.items():
    for player_record in players_list:
      player_record['Season'] = season
      player_record['Team'] = team_name
      all_rows.append(player_record)

df_auction_sold_players_details = pd.DataFrame(all_rows)
df_auction_sold_players_details['Base Price'] = pd.to_numeric(df_auction_sold_players_details['Base Price'], errors='coerce')
df_auction_sold_players_details['Winning Bid'] = pd.to_numeric(df_auction_sold_players_details['Winning Bid'], errors='coerce')

print("✅ Successfully Fetched Auction Sold Players Details")

Fetching IPL Season 2026 Stats...
Successfully fetched the webpage
Fetching IPL Season 2025 Stats...
Successfully fetched the webpage
Fetching IPL Season 2024 Stats...
Successfully fetched the webpage
Fetching IPL Season 2023 Stats...
Successfully fetched the webpage
Fetching IPL Season 2022 Stats...
Successfully fetched the webpage
Fetching IPL Season 2021 Stats...
Successfully fetched the webpage
Fetching IPL Season 2020 Stats...
Successfully fetched the webpage
Fetching IPL Season 2019 Stats...
Successfully fetched the webpage
Fetching IPL Season 2018 Stats...
Successfully fetched the webpage
Fetching IPL Season 2017 Stats...
Successfully fetched the webpage
Fetching IPL Season 2016 Stats...
Successfully fetched the webpage
Fetching IPL Season 2015 Stats...
Successfully fetched the webpage
Fetching IPL Season 2014 Stats...
Successfully fetched the webpage
Fetching IPL Season 2013 Stats...
Successfully fetched the webpage
Fetching IPL Season 2012 Stats...
Successfully fetched the web

### Matches Data Fetch

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d")

#Events Detail
print("Fetching All Seasons Details...")
data = fetch_data_y(years_url, None, "oncomptetion(", None)
Division = fetch_data_y(years_url, None, "oncomptetion(", 'division')
Competition = fetch_data_y(years_url, None, "oncomptetion(", 'competition')
for recordsA, recordB in zip(Division,Competition):
  EventList.append(extract_seasonsDetail(recordsA,recordB))
df_eventrecords = pd.DataFrame(EventList)
print("Successfully Fetching All Seasons Details!")

#Players Carrrier Detail
print("Fetching Players Carrrier Details...")
data = fetch_data_y(playersCarrier_url, None, "onIPLAllPlayerCarrierStats(", 'Batting')
for player in data:
    PlayerBattingCarrier.append(extract_battingCarrier(player))
data = fetch_data_y(playersCarrier_url, None, "onIPLAllPlayerCarrierStats(", 'Bowling')
for player in data:
    PlayerBowlingCarrier.append(extract_bowlingCarrier(player))

df_PlayersBattingCarrier = pd.DataFrame(PlayerBattingCarrier)
df_PlayersBowlingCarrier = pd.DataFrame(PlayerBowlingCarrier)
print("Successfully Fetching Players Carrrier Details!")

Fetching All Seasons Details...
Successfully Fetching All Seasons Details!
Fetching Players Carrrier Details...
Successfully Fetching Players Carrrier Details!


In [ ]:
#Season
for year in YearsList:

    #Season Group Standings
    print(f"Fetching Group Standing Data For CompetitionID: {SeasonList[YearsList.index(year)]}")
    data = fetch_data_y(groupstandings_url, year, "ongroupstandings(", 'points')

    for stand in data:
        GroupStandings.append(extract_standings(stand))

    df_groupstandings = pd.DataFrame(GroupStandings)
    df_groupstandings['Matches'] = pd.to_numeric(df_groupstandings['Matches'], errors='coerce')
    df_groupstandings['Wins'] = pd.to_numeric(df_groupstandings['Wins'], errors='coerce')
    df_groupstandings['Loss'] = pd.to_numeric(df_groupstandings['Loss'], errors='coerce')
    df_groupstandings['Tied'] = pd.to_numeric(df_groupstandings['Tied'], errors='coerce')
    df_groupstandings['NoResult'] = pd.to_numeric(df_groupstandings['NoResult'], errors='coerce')
    df_groupstandings['Points'] = pd.to_numeric(df_groupstandings['Points'], errors='coerce')
    df_groupstandings['Draw'] = pd.to_numeric(df_groupstandings['Draw'], errors='coerce')
    df_groupstandings['LeadBy'] = pd.to_numeric(df_groupstandings['LeadBy'], errors='coerce')
    df_groupstandings['Deficit'] = pd.to_numeric(df_groupstandings['Deficit'], errors='coerce')
    print(f"Successfully Completed Fetching Group Standing Data For CompetitionID: {SeasonList[YearsList.index(year)]}")

    #Season Orange Cap Standings
    print(f"Fetching Orange Cap Data For CompetitionID: {SeasonList[YearsList.index(year)]}")
    data = fetch_data_y(orangecap_standings_url, year, "ontoprunsscorers(", 'toprunsscorers')

    for orange in data:
        OrangeCap.append(extract_orangecap(orange))

    df_orangecap = pd.DataFrame(OrangeCap)
    print(f"Successfully Completed Fetching Orange Cap Data For CompetitionID: {SeasonList[YearsList.index(year)]}")

    #Season Purple Cap Standings
    print(f"Fetching Purple Cap Data For CompetitionID: {SeasonList[YearsList.index(year)]}")
    data= fetch_data_y(purplecap_standings_url, year, "onmostwickets(", 'mostwickets')

    for purple in data:
        PurpleCap.append(extract_purplecap(purple))

    df_purplecap = pd.DataFrame(PurpleCap)
    print(f"Successfully Completed Fetching Purple Cap Data For CompetitionID: {SeasonList[YearsList.index(year)]}")

    #MatchSummary
    print(f"Initiated Extraction of Matchsummary Data For CompetitionID: {SeasonList[YearsList.index(year)]}")
    match_list = fetch_data_y(base_schedule_url, year, "MatchSchedule(", 'Matchsummary')

    for m in match_list:
      MatchID = m.get('MatchID')

      if not MatchID:
        continue

      match_summary_list = fetch_data_m(base_summary_url, MatchID, "onScoringMatchsummary(", 'MatchSummary')
      for mat in match_summary_list:
        MatchSummary.append(extract_matchsummary(m,mat))

    df_matchsummary = pd.DataFrame(MatchSummary)
    if not df_matchsummary.empty:
        df_matchsummary['MATCH_NO_OF_OVERS'] = pd.to_numeric(df_matchsummary['MATCH_NO_OF_OVERS'], errors='coerce')
        df_matchsummary['RevisedOver'] = pd.to_numeric(df_matchsummary['RevisedOver'], errors='coerce')
        df_matchsummary['RevisedTarget'] = pd.to_numeric(df_matchsummary['RevisedTarget'], errors='coerce')
        df_matchsummary['Target'] = pd.to_numeric(df_matchsummary['Target'], errors='coerce')
        df_matchsummary['MOMRuns'] = pd.to_numeric(df_matchsummary['MOMRuns'], errors='coerce')
        df_matchsummary['MOMBalls'] = pd.to_numeric(df_matchsummary['MOMBalls'], errors='coerce')
        df_matchsummary['MOMWicket'] = pd.to_numeric(df_matchsummary['MOMWicket'], errors='coerce')
        df_matchsummary['MOMRC'] = pd.to_numeric(df_matchsummary['MOMRC'], errors='coerce')

    print(f"Completed Extraction of Matchsummary Data For CompetitionID: {SeasonList[YearsList.index(year)]}")

    #Fetch Each Seasons Matches Details
    print(f"Initiated Extraction of Match Data For CompetitionID: {SeasonList[YearsList.index(year)]}")
    for match in match_list:
      MatchID = match.get('MatchID')

      if not MatchID:
        continue

      print('Initiate Match_id:',MatchID,"Data-Fetch\n")
      inn_1=innings(MatchID,base_innings_url_1,'Innings1')
      inn_2=innings(MatchID,base_innings_url_2,'Innings2')
      squad_data = squad_fun(MatchID,match_squad_url)

      #Team-1 Sqaud
      if squad_data and squad_data.get('squadA'):
        for player in squad_data['squadA']:
          Squad.append(extract_squad(MatchID,player))

      #Team-2 Sqaud
      if squad_data and squad_data.get('squadB'):
        for player in squad_data['squadB']:
          Squad.append(extract_squad(MatchID,player))

      df_squad = pd.DataFrame(Squad)
      if not df_squad.empty:
        df_squad['PlayerID'] = df_squad['PlayerID'].astype(str)
        df_squad['TeamID'] = df_squad['TeamID'].astype(str)


      if not inn_1 or not inn_2:
        continue

      #BattingScoreCard
      for record in inn_1['Innings1']['BattingCard']:
        BattingCard.append(extract_battingCard(record))
      for record in inn_2['Innings2']['BattingCard']:
        BattingCard.append(extract_battingCard(record))
      df_battingCard = pd.DataFrame(BattingCard)
      df_battingCard['PlayingOrder'] = pd.to_numeric(df_battingCard['PlayingOrder'], errors='coerce')
      df_battingCard['MatchPlayingOrder'] = pd.to_numeric(df_battingCard['MatchPlayingOrder'], errors='coerce')
      df_battingCard['Runs'] = pd.to_numeric(df_battingCard['Runs'], errors='coerce')
      df_battingCard['Balls'] = pd.to_numeric(df_battingCard['Balls'], errors='coerce')
      df_battingCard['DotBalls'] = pd.to_numeric(df_battingCard['DotBalls'], errors='coerce')
      df_battingCard['Ones'] = pd.to_numeric(df_battingCard['Ones'], errors='coerce')
      df_battingCard['Twos'] = pd.to_numeric(df_battingCard['Twos'], errors='coerce')
      df_battingCard['Threes'] = pd.to_numeric(df_battingCard['Threes'], errors='coerce')
      df_battingCard['Fours'] = pd.to_numeric(df_battingCard['Fours'], errors='coerce')
      df_battingCard['Sixes'] = pd.to_numeric(df_battingCard['Sixes'], errors='coerce')
      df_battingCard['WicketNo'] = pd.to_numeric(df_battingCard['WicketNo'], errors='coerce')
      df_battingCard['MinOver'] = pd.to_numeric(df_battingCard['MinOver'], errors='coerce')
      df_battingCard['MinStrikerOver'] = pd.to_numeric(df_battingCard['MinStrikerOver'], errors='coerce')

      #BowlingScoreCard
      for record in inn_1['Innings1']['BowlingCard']:
        BowlingCard.append(extract_bowlingCard(record))
      for record in inn_2['Innings2']['BowlingCard']:
        BowlingCard.append(extract_bowlingCard(record))
      df_bowlingCard = pd.DataFrame(BowlingCard)
      df_bowlingCard['Overs'] = pd.to_numeric(df_bowlingCard['Overs'], errors='coerce')
      df_bowlingCard['Maidens'] = pd.to_numeric(df_bowlingCard['Maidens'], errors='coerce')
      df_bowlingCard['Runs'] = pd.to_numeric(df_bowlingCard['Runs'], errors='coerce')
      df_bowlingCard['Wickets'] = pd.to_numeric(df_bowlingCard['Wickets'], errors='coerce')
      df_bowlingCard['Wides'] = pd.to_numeric(df_bowlingCard['Wides'], errors='coerce')
      df_bowlingCard['NoBalls'] = pd.to_numeric(df_bowlingCard['NoBalls'], errors='coerce')
      df_bowlingCard['Economy'] = pd.to_numeric(df_bowlingCard['Economy'], errors='coerce')
      df_bowlingCard['BowlingOrder'] = pd.to_numeric(df_bowlingCard['BowlingOrder'], errors='coerce')
      df_bowlingCard['TotalLegalBallsBowled'] = pd.to_numeric(df_bowlingCard['TotalLegalBallsBowled'], errors='coerce')
      df_bowlingCard['ScoringBalls'] = pd.to_numeric(df_bowlingCard['ScoringBalls'], errors='coerce')
      df_bowlingCard['DotBalls'] = pd.to_numeric(df_bowlingCard['DotBalls'], errors='coerce')
      df_bowlingCard['Ones'] = pd.to_numeric(df_bowlingCard['Ones'], errors='coerce')
      df_bowlingCard['Twos'] = pd.to_numeric(df_bowlingCard['Twos'], errors='coerce')
      df_bowlingCard['Threes'] = pd.to_numeric(df_bowlingCard['Threes'], errors='coerce')
      df_bowlingCard['Fours'] = pd.to_numeric(df_bowlingCard['Fours'], errors='coerce')
      df_bowlingCard['Sixes'] = pd.to_numeric(df_bowlingCard['Sixes'], errors='coerce')

      #Extras
      for record in inn_1['Innings1']['Extras']:
        Extras.append(extract_extras(record))
      for record in inn_2['Innings2']['Extras']:
        Extras.append(extract_extras(record))
      df_Extras = pd.DataFrame(Extras)

      #Fall Of Wickets
      for record in inn_1['Innings1']['FallOfWickets']:
        FallOfWickets.append(extract_FallOfWickets(record))
      for record in inn_2['Innings2']['FallOfWickets']:
        FallOfWickets.append(extract_FallOfWickets(record))
      df_FallOfWickets = pd.DataFrame(FallOfWickets)
      df_FallOfWickets['InningsNo'] = pd.to_numeric(df_FallOfWickets['InningsNo'], errors='coerce')
      df_FallOfWickets['FallScore'] = pd.to_numeric(df_FallOfWickets['FallScore'], errors='coerce')
      df_FallOfWickets['FallWickets'] = pd.to_numeric(df_FallOfWickets['FallWickets'], errors='coerce')
      df_FallOfWickets['FallOvers'] = pd.to_numeric(df_FallOfWickets['FallOvers'], errors='coerce')

      #Wagon Wheel
      for record in inn_1['Innings1']['WagonWheel']:
        WagonWheel.append(extract_WagonWheel(MatchID,record))
      for record in inn_2['Innings2']['WagonWheel']:
        WagonWheel.append(extract_WagonWheel(MatchID,record))
      df_WagonWheel = pd.DataFrame(WagonWheel)

      #Partnership Scores
      for record in inn_1['Innings1']['PartnershipScores']:
        PartnershipScores.append(extract_PartnershipScores(record))
      for record in inn_2['Innings2']['PartnershipScores']:
        PartnershipScores.append(extract_PartnershipScores(record))
      df_PartnershipScores = pd.DataFrame(PartnershipScores)

      #Partnership Break
      for record in inn_1['Innings1']['PartnershipBreak']:
        PartnershipBreak.append(extract_PartnershipBreak(MatchID,record))
      for record in inn_2['Innings2']['PartnershipBreak']:
        PartnershipBreak.append(extract_PartnershipBreak(MatchID,record))
      df_PartnershipBreak = pd.DataFrame(PartnershipBreak)
      df_PartnershipBreak['InningsNo'] = pd.to_numeric(df_PartnershipBreak['InningsNo'], errors='coerce')
      df_PartnershipBreak['OverNo'] = pd.to_numeric(df_PartnershipBreak['OverNo'], errors='coerce')
      df_PartnershipBreak['WicketNo'] = pd.to_numeric(df_PartnershipBreak['WicketNo'], errors='coerce')

      #Manhattan Graph
      for record in inn_1['Innings1']['ManhattanGraph']:
        ManhattanGraph.append(extract_ManhattanGraph(MatchID,record))
      for record in inn_2['Innings2']['ManhattanGraph']:
        ManhattanGraph.append(extract_ManhattanGraph(MatchID,record))
      df_ManhattanGraph = pd.DataFrame(ManhattanGraph)

      #Manhattan Wickets
      for record in inn_1['Innings1']['ManhattanWickets']:
        ManhattanWickets.append(extract_ManhattanWickets(MatchID,record))
      for record in inn_2['Innings2']['ManhattanWickets']:
        ManhattanWickets.append(extract_ManhattanWickets(MatchID,record))
      df_ManhattanWickets = pd.DataFrame(ManhattanWickets)
      df_ManhattanWickets['InningsNo'] = pd.to_numeric(df_ManhattanWickets['InningsNo'], errors='coerce')
      df_ManhattanWickets['OverNo'] = pd.to_numeric(df_ManhattanWickets['OverNo'], errors='coerce')
      df_ManhattanWickets['BatsmanRuns'] = pd.to_numeric(df_ManhattanWickets['BatsmanRuns'], errors='coerce')
      df_ManhattanWickets['BatsmanBalls'] = pd.to_numeric(df_ManhattanWickets['BatsmanBalls'], errors='coerce')
      df_ManhattanWickets['BatsManRuns2'] = pd.to_numeric(df_ManhattanWickets['BatsManRuns2'], errors='coerce')

      #Over History
      for record in inn_1['Innings1']['OverHistory']:
        OverHistory.append(extract_OverHistory(record))
      for record in inn_2['Innings2']['OverHistory']:
        OverHistory.append(extract_OverHistory(record))
      df_OverHistory = pd.DataFrame(OverHistory)

      #Wagon Wheel Summary
      for record in inn_1['Innings1']['WagonWheelSummary']:
        WagonWheelSummary.append(extract_WagonWheelSummary(record))
      for record in inn_2['Innings2']['WagonWheelSummary']:
        WagonWheelSummary.append(extract_WagonWheelSummary(record))
      df_WagonWheelSummary = pd.DataFrame(WagonWheelSummary)

      #Batting Head-2-Head
      for record in inn_1['Innings1']['battingheadtohead']:
        Battingheadtohead.append(extract_battingheadtohead(MatchID,record))
      for record in inn_2['Innings2']['battingheadtohead']:
        Battingheadtohead.append(extract_battingheadtohead(MatchID,record))
      df_Battingheadtohead = pd.DataFrame(Battingheadtohead)

      #Bowling Head-2-Head
      for record in inn_1['Innings1']['bowlingheadtohead']:
        Bowlingheadtohead.append(extract_bowlingheadtohead(record))
      for record in inn_2['Innings2']['bowlingheadtohead']:
        Bowlingheadtohead.append(extract_bowlingheadtohead(record))
      df_Bowlingheadtohead = pd.DataFrame(Bowlingheadtohead)

      #Teams Head-2-Head
      data = fetch_data_m(team_h2h_url, MatchID, "onMatchHTH(", 'HeadToHead')

      for team in data:
        Team_h2h.append(extract_team_h2h(team))

      df_team_h2h = pd.DataFrame(Team_h2h)

      print('Completed Match_id:',MatchID,"Data-Fetch\n")

    print(f"Completed Extraction of Match Data For CompetitionID: {SeasonList[YearsList.index(year)]}")

Streaming output truncated to the last 5000 lines.
Initiated Extraction of Match Data For CompetitionID: 2020
Initiate Match_id: 10831 Data-Fetch

Could not fetch Innings1 for matchID 10831
Could not fetch Innings2 for matchID 10831
Could not fetch Squad for matchID 10831
Initiate Match_id: 10829 Data-Fetch

Could not fetch Innings1 for matchID 10829
Could not fetch Innings2 for matchID 10829
Could not fetch Squad for matchID 10829
Initiate Match_id: 10827 Data-Fetch

Could not fetch Innings1 for matchID 10827
Could not fetch Innings2 for matchID 10827
Could not fetch Squad for matchID 10827
Initiate Match_id: 10826 Data-Fetch

Could not fetch Innings1 for matchID 10826
Could not fetch Innings2 for matchID 10826
Could not fetch Squad for matchID 10826
Initiate Match_id: 10823 Data-Fetch

Could not fetch Innings1 for matchID 10823
Could not fetch Innings2 for matchID 10823
Could not fetch Squad for matchID 10823
Initiate Match_id: 10822 Data-Fetch

Could not fetch Innings1 for matchID 1

###Upload To Snowflake

In [ ]:
#Raw Table Names

##Carrier Tables
first_schema = "CARRIER"
eventrecords = "RAW_SEASONS_DETAIL"
PlayersBattingCarrier = "RAW_PLAYERS_BATTING_CAREER"
PlayersBowlingCarrier = "RAW_PLAYERS_BOWLING_CAREER"

##Seasons Tables
second_schema = "SEASON"
auction_stats = "RAW_AUCTION_STATS"
auction_team_stats = "RAW_AUCTION_TEAM_STATS"
player_auction_list = "RAW_PLAYER_AUCTION_LIST"
auction_sold_players_details = "RAW_AUCTION_SOLD_PLAYERS_DETAILS"
matchsummary = "RAW_MATCH_SUMMARY"
groupstandings = "RAW_GROUP_STANDINGS"
orangecap = "RAW_ORANGECAP_STANDINGS"
purplecap = "RAW_PURPLECAP_STANDINGS"

##Matches Tables
third_schema = "MATCH"
battingCard = "RAW_BATTING_CARD"
bowlingCard = "RAW_BOWLING_CARD"
extras = "RAW_MATCH_EXTRAS"
FallOfWickets = "RAW_FALL_OF_WICKETS"
battingh2h = "RAW_BATTING_HEAD_TO_HEAD"
bowlingh2h = "RAW_BOWLING_HEAD_TO_HEAD"
squad = "RAW_SQUAD"
team_h2h = "RAW_TEAM_H2H"
PartnershipScores = "RAW_PARTNERSHIP_SCORES"
PartnershipBreake = "RAW_PARTNERSHIP_BREAK"
ManhattanGraph = "RAW_MANHATTAN_GRAPH"
ManhattanWickets = "RAW_MANHATTAN_WICKETS"
OverHistory = "RAW_OVER_HISTORY"
WagonWheel = "RAW_WAGON_WHEEL"
WagonWheelSummary = "RAW_WAGON_WHEEL_SUMMARY"

In [ ]:
# Upload To Snowflake

##Carrier Schema Tables
print("Starting to Create Carrier Schema Tables...",end='\n')
cur.execute(f"DROP TABLE IF EXISTS {first_schema}.{eventrecords}")
if not df_eventrecords.empty:
  df_eventrecords = df_eventrecords.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_eventrecords, table_name=eventrecords, schema=first_schema, auto_create_table=True)
  if success:
    print(f"Table {eventrecords} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {eventrecords}",end='\n')
else:
  print(f"DataFrame {eventrecords} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {first_schema}.{PlayersBattingCarrier}")
if not df_PlayersBattingCarrier.empty:
  df_PlayersBattingCarrier = df_PlayersBattingCarrier.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_PlayersBattingCarrier, table_name=PlayersBattingCarrier, schema=first_schema, auto_create_table=True)
  if success:
    print(f"Table {PlayersBattingCarrier} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {PlayersBattingCarrier}",end='\n')
else:
  print(f"DataFrame {PlayersBattingCarrier} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {first_schema}.{PlayersBowlingCarrier}")
if not df_PlayersBowlingCarrier.empty:
  df_PlayersBowlingCarrier = df_PlayersBowlingCarrier.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_PlayersBowlingCarrier, table_name=PlayersBowlingCarrier, schema=first_schema, auto_create_table=True)
  if success:
    print(f"Table {PlayersBowlingCarrier} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {PlayersBowlingCarrier}",end='\n')
else:
  print(f"DataFrame {PlayersBowlingCarrier} is empty, skipping table creation.",end='\n')

##Season Schema Tables
print("Starting to Create Season Schema Tables...",end='\n')
cur.execute(f"DROP TABLE IF EXISTS {second_schema}.{auction_stats}")
if not df_auction_stats.empty:
  df_auction_stats = df_auction_stats.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_auction_stats, table_name=auction_stats, schema=second_schema, auto_create_table=True)
  if success:
    print(f"Table {auction_stats} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {auction_stats}",end='\n')
else:
  print(f"DataFrame {auction_stats} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {second_schema}.{auction_team_stats}")
if not df_auction_team_stats.empty:
  df_auction_team_stats = df_auction_team_stats.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_auction_team_stats, table_name=auction_team_stats, schema=second_schema, auto_create_table=True)
  if success:
    print(f"Table {auction_team_stats} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {auction_team_stats}",end='\n')
else:
  print(f"DataFrame {auction_team_stats} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {second_schema}.{player_auction_list}")
if not df_player_auction_list.empty:
  df_player_auction_list = df_player_auction_list.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_player_auction_list, table_name=player_auction_list, schema=second_schema, auto_create_table=True)
  if success:
    print(f"Table {player_auction_list} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {player_auction_list}",end='\n')
else:
  print(f"DataFrame {player_auction_list} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {second_schema}.{auction_sold_players_details}")
if not df_auction_sold_players_details.empty:
  df_auction_sold_players_details = df_auction_sold_players_details.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_auction_sold_players_details, table_name=auction_sold_players_details, schema=second_schema, auto_create_table=True)
  if success:
    print(f"Table {auction_sold_players_details} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {auction_sold_players_details}",end='\n')
else:
  print(f"DataFrame {auction_sold_players_details} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {second_schema}.{matchsummary}")
if not df_matchsummary.empty:
  df_matchsummary = df_matchsummary.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_matchsummary, table_name=matchsummary, schema=second_schema, auto_create_table=True)
  if success:
    print(f"Table {matchsummary} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {matchsummary}",end='\n')
else:
  print(f"DataFrame {matchsummary} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {second_schema}.{groupstandings}")
if not df_groupstandings.empty:
  df_groupstandings = df_groupstandings.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_groupstandings, table_name=groupstandings, schema=second_schema, auto_create_table=True)
  if success:
    print(f"Table {groupstandings} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {groupstandings}",end='\n')
else:
  print(f"DataFrame {groupstandings} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {second_schema}.{orangecap}")
if not df_orangecap.empty:
  df_orangecap = df_orangecap.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_orangecap, table_name=orangecap, schema=second_schema, auto_create_table=True)
  if success:
    print(f"Table {orangecap} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {orangecap}",end='\n')
else:
  print(f"DataFrame {orangecap} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {second_schema}.{purplecap}")
if not df_purplecap.empty:
  df_purplecap = df_purplecap.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_purplecap, table_name=purplecap, schema=second_schema, auto_create_table=True)
  if success:
    print(f"Table {purplecap} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {purplecap}",end='\n')
else:
  print(f"DataFrame {purplecap} is empty, skipping table creation.",end='\n')


##Match Schema Tables
print("Starting to Create Match Schema Tables...",end='\n')
cur.execute(f"DROP TABLE IF EXISTS {third_schema}.{battingCard}")
if not df_battingCard.empty:
  df_battingCard = df_battingCard.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_battingCard, table_name=battingCard, schema=third_schema, auto_create_table=True)
  if success:
    print(f"Table {battingCard} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {battingCard}",end='\n')
else:
  print(f"DataFrame {battingCard} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {third_schema}.{bowlingCard}")
if not df_bowlingCard.empty:
  df_bowlingCard = df_bowlingCard.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_bowlingCard, table_name=bowlingCard, schema=third_schema, auto_create_table=True)
  if success:
    print(f"Table {bowlingCard} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {bowlingCard}",end='\n')
else:
  print(f"DataFrame {bowlingCard} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {third_schema}.{extras}")
if not df_Extras.empty:
  df_Extras = df_Extras.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_Extras, table_name=extras, schema=third_schema, auto_create_table=True)
  if success:
    print(f"Table {extras} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {extras}",end='\n')
else:
  print(f"DataFrame {extras} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {third_schema}.{FallOfWickets}")
if not df_FallOfWickets.empty:
  df_FallOfWickets = df_FallOfWickets.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_FallOfWickets, table_name=FallOfWickets, schema=third_schema, auto_create_table=True)
  if success:
    print(f"Table {FallOfWickets} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {FallOfWickets}",end='\n')
else:
  print(f"DataFrame {FallOfWickets} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {third_schema}.{battingh2h}")
if not df_Battingheadtohead.empty:
  df_Battingheadtohead = df_Battingheadtohead.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_Battingheadtohead, table_name=battingh2h, schema=third_schema, auto_create_table=True)
  if success:
    print(f"Table {battingh2h} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {battingh2h}",end='\n')
else:
  print(f"DataFrame {battingh2h} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {third_schema}.{bowlingh2h}")
if not df_Bowlingheadtohead.empty:
  df_Bowlingheadtohead = df_Bowlingheadtohead.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_Bowlingheadtohead, table_name=bowlingh2h, schema=third_schema, auto_create_table=True)
  if success:
    print(f"Table {bowlingh2h} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {bowlingh2h}",end='\n')
else:
  print(f"DataFrame {bowlingh2h} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {third_schema}.{squad}")
if not df_squad.empty:
  df_squad = df_squad.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_squad, table_name=squad, schema=third_schema, auto_create_table=True)
  if success:
    print(f"Table {squad} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {squad}",end='\n')
else:
  print(f"DataFrame {squad} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {third_schema}.{team_h2h}")
if not df_team_h2h.empty:
  df_team_h2h = df_team_h2h.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_team_h2h, table_name=team_h2h, schema=third_schema, auto_create_table=True)
  if success:
    print(f"Table {team_h2h} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {team_h2h}",end='\n')
else:
  print(f"DataFrame {team_h2h} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {third_schema}.{PartnershipScores}")
if not df_PartnershipScores.empty:
  df_PartnershipScores = df_PartnershipScores.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_PartnershipScores, table_name=PartnershipScores, schema=third_schema, auto_create_table=True)
  if success:
    print(f"Table {PartnershipScores} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {PartnershipScores}",end='\n')
else:
  print(f"DataFrame {PartnershipScores} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {third_schema}.{PartnershipBreake}")
if not df_PartnershipBreak.empty:
  df_PartnershipBreak = df_PartnershipBreak.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_PartnershipBreak, table_name=PartnershipBreake, schema=third_schema, auto_create_table=True)
  if success:
    print(f"Table {PartnershipBreake} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {PartnershipBreake}",end='\n')
else:
  print(f"DataFrame {PartnershipBreake} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {third_schema}.{ManhattanGraph}")
if not df_ManhattanGraph.empty:
  df_ManhattanGraph = df_ManhattanGraph.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_ManhattanGraph, table_name=ManhattanGraph, schema=third_schema, auto_create_table=True)
  if success:
    print(f"Table {ManhattanGraph} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {ManhattanGraph}",end='\n')
else:
  print(f"DataFrame {ManhattanGraph} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {third_schema}.{ManhattanWickets}")
if not df_ManhattanWickets.empty:
  df_ManhattanWickets = df_ManhattanWickets.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_ManhattanWickets, table_name=ManhattanWickets, schema=third_schema, auto_create_table=True)
  if success:
    print(f"Table {ManhattanWickets} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {ManhattanWickets}",end='\n')
else:
  print(f"DataFrame {ManhattanWickets} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {third_schema}.{OverHistory}")
if not df_OverHistory.empty:
  df_OverHistory = df_OverHistory.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_OverHistory, table_name=OverHistory, schema=third_schema, auto_create_table=True)
  if success:
    print(f"Table {OverHistory} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {OverHistory}",end='\n')
else:
  print(f"DataFrame {OverHistory} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {third_schema}.{WagonWheel}")
if not df_WagonWheel.empty:
  df_WagonWheel = df_WagonWheel.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_WagonWheel, table_name=WagonWheel, schema=third_schema, auto_create_table=True)
  if success:
    print(f"Table {WagonWheel} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {WagonWheel}",end='\n')
else:
  print(f"DataFrame {WagonWheel} is empty, skipping table creation.",end='\n')

cur.execute(f"DROP TABLE IF EXISTS {third_schema}.{WagonWheelSummary}")
if not df_WagonWheelSummary.empty:
  df_WagonWheelSummary = df_WagonWheelSummary.astype(str)
  success, nchunks, nrows, _ = write_pandas(conn, df_WagonWheelSummary, table_name=WagonWheelSummary, schema=third_schema, auto_create_table=True)
  if success:
    print(f"Table {WagonWheelSummary} is Created With Total {nrows} Rows",end='\n')
  else:
    print(f"Failed to Create Table {WagonWheelSummary}",end='\n')
else:
  print(f"DataFrame {WagonWheelSummary} is empty, skipping table creation.",end='\n')

print("✅ All Tables created with all required attributes")

Starting to Create Carrier Schema Tables...
Table RAW_SEASONS_DETAIL is Created With Total 19 Rows
Table RAW_PLAYERS_BATTING_CAREER is Created With Total 467 Rows
Table RAW_PLAYERS_BOWLING_CAREER is Created With Total 471 Rows
Starting to Create Season Schema Tables...
Table RAW_AUCTION_STATS is Created With Total 19 Rows
Table RAW_AUCTION_TEAM_STATS is Created With Total 40 Rows
Table RAW_PLAYER_AUCTION_LIST is Created With Total 8859 Rows
Table RAW_AUCTION_SOLD_PLAYERS_DETAILS is Created With Total 908 Rows
Table RAW_MATCH_SUMMARY is Created With Total 296 Rows
Table RAW_GROUP_STANDINGS is Created With Total 166 Rows
Table RAW_ORANGECAP_STANDINGS is Created With Total 2664 Rows
Table RAW_PURPLECAP_STANDINGS is Created With Total 1684 Rows
Starting to Create Match Schema Tables...
Table RAW_BATTING_CARD is Created With Total 6797 Rows
Table RAW_BOWLING_CARD is Created With Total 3462 Rows
Table RAW_MATCH_EXTRAS is Created With Total 580 Rows
Table RAW_FALL_OF_WICKETS is Created With T